In [1]:
from pathlib import Path
import os

import pandas as pd
import numpy as np
import psycopg
from dotenv import load_dotenv

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
ENV_PATH = PROJECT_ROOT / ".env"

FINAL_EMBEDDINGS_PATH = DATA_DIR / "final_embeddings.parquet"

load_dotenv(ENV_PATH)

DATABASE_URL = os.getenv("DATABASE_URL", "").strip()
APP_TABLE_NAME = "course_embeddings_app"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_EMBEDDINGS_PATH exists:", FINAL_EMBEDDINGS_PATH.exists())
print("DATABASE_URL loaded:", bool(DATABASE_URL))
print("APP_TABLE_NAME:", APP_TABLE_NAME)

PROJECT_ROOT: C:\Users\user\Documents\thesis
FINAL_EMBEDDINGS_PATH exists: True
DATABASE_URL loaded: True
APP_TABLE_NAME: course_embeddings_app


In [3]:
final_embeddings_df = pd.read_parquet(FINAL_EMBEDDINGS_PATH).copy()

required_columns = [
    "course_unit_code",
    "course_name",
    "final_embedding_text",
    "embedding_vector",
]

missing = [c for c in required_columns if c not in final_embeddings_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

print(final_embeddings_df.shape)
display(final_embeddings_df.head(3))

(50, 14)


,course_unit_code,course_name,final_embedding_text,final_model_name,final_openai_model_id,final_text_variant,final_similarity_metric,final_selection_rule,final_locked_threshold,embedding_dim,embedding_generation_seconds,raw_embedding_norm,normalized_embedding_norm,embedding_vector
0,TT00CD70,Data Networks,Data Networks\nYou learn the structure and pro...,openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,3.002128,1.000048,1.0,"[-0.02810533344745636, 0.020735694095492363, -..."
1,TT00CD71,Linux Basics,"Linux Basics\nAfter completing the course, you...",openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,3.002128,0.999847,1.0,"[-0.013971555046737194, 0.03848854452371597, -..."
2,TT00CD72,Servers and containers,Servers and containers\nYou understand the fun...,openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,3.002128,0.999609,1.0,"[-0.028056597337126732, 0.029399896040558815, ..."


In [4]:
def quote_identifier(identifier: str) -> str:
    if not isinstance(identifier, str) or not identifier.strip():
        raise ValueError("SQL identifier must be a non-empty string.")
    return '"' + identifier.replace('"', '""') + '"'


def embedding_to_pgvector_text(vector) -> str:
    if isinstance(vector, np.ndarray):
        vector = vector.tolist()
    elif not isinstance(vector, list):
        vector = list(vector)

    return "[" + ",".join(str(float(x)) for x in vector) + "]"

In [5]:
embedding_dim = len(final_embeddings_df.iloc[0]["embedding_vector"])
table_name_sql = quote_identifier(APP_TABLE_NAME)

create_sql = f"""
CREATE EXTENSION IF NOT EXISTS vector;

DROP TABLE IF EXISTS {table_name_sql};

CREATE TABLE {table_name_sql} (
    course_unit_code TEXT PRIMARY KEY,
    course_name TEXT NOT NULL,
    final_embedding_text TEXT NOT NULL,
    embedding_vector vector({embedding_dim}) NOT NULL
);
"""

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(create_sql)
    conn.commit()

print("Created table:", APP_TABLE_NAME)
print("Embedding dimension:", embedding_dim)

Created table: course_embeddings_app
Embedding dimension: 3072


In [6]:
insert_sql = f"""
INSERT INTO {table_name_sql} (
    course_unit_code,
    course_name,
    final_embedding_text,
    embedding_vector
)
VALUES (%s, %s, %s, %s::vector);
"""

records = []
for _, row in final_embeddings_df.iterrows():
    records.append(
        (
            str(row["course_unit_code"]),
            str(row["course_name"]),
            str(row["final_embedding_text"]),
            embedding_to_pgvector_text(row["embedding_vector"]),
        )
    )

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.executemany(insert_sql, records)
    conn.commit()

print("Inserted rows:", len(records))

Inserted rows: 50


In [7]:
check_sql = f"""
SELECT
    course_unit_code,
    course_name
FROM {table_name_sql}
ORDER BY course_unit_code
LIMIT 10;
"""

count_sql = f"SELECT COUNT(*) FROM {table_name_sql};"

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(count_sql)
        row_count = cur.fetchone()[0]

        cur.execute(check_sql)
        rows = cur.fetchall()

print("Row count:", row_count)
display(pd.DataFrame(rows, columns=["course_unit_code", "course_name"]))

Row count: 50


,course_unit_code,course_name
0,TE00BR86,Back-End Development
1,TE00CN56,Internet Networks and Security
2,TE00CS88,Introduction to Programming
3,TE00CS89,Databases
4,TT00CD70,Data Networks
5,TT00CD71,Linux Basics
6,TT00CD72,Servers and containers
7,TT00CD75,Windows Basics
8,TT00CD76,Scripting and Automatization
9,TT00CD77,Basics of Programming
